# 코스피지수 예측 — 선형회귀, 릿지, 라쏘, 다항식 회귀

**1일차 · 회귀분석 실습** (원본 *코스피지수_예측.ipynb*의 보강판)

거시·금융 지표 29개로 코스피지수를 예측합니다. 데이터는 160행(2003년 5월 ~ 2016년 8월)이고, 각 행을 독립 관측치로 보아 80/20으로 무작위 분할합니다.

**실습 내용:** OLS → 릿지 → 라쏘 → 다항식 특성 → 규제를 건 다항식 모델을 8절에서 한 표로 비교합니다.

**원본 대비 달라진 점**
1. 데이터 파일을 UTF-8로 바꾸고 `날짜` 열을 추가했습니다 (`KOSPI_Index_KO.csv`).
2. `r2_score`는 `r2_score(y_true, y_pred)` 순서로 호출합니다. 원본의 `r2(y_pred, y_test)`는 인자 순서가 바뀌어 있었는데, R²는 **대칭이 아닙니다**.

## 0. 준비

In [1]:
import os
FILE = 'KOSPI_Index_KO.csv'

# Colab: 안내가 나오면 KOSPI_Index_KO.csv를 업로드하세요 (또는 왼쪽 파일 패널로 끌어다 놓기)
if not os.path.exists(FILE):
    try:
        from google.colab import files
        print('업로드할 파일:', FILE)
        files.upload()
    except ImportError:
        raise FileNotFoundError(f'{os.getcwd()}에 {FILE}이 없습니다')

In [2]:
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 12)

# 그래프 한글 표시 (Colab에서 한글이 깨지면 아래 두 줄의 주석을 풀고 런타임을 다시 시작)
# !apt-get -qq install fonts-nanum
# plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터 불러오기와 확인

In [3]:
kospi = pd.read_csv(FILE, encoding='utf-8-sig')   # 원본 cp949 파일은 encoding='cp949'
kospi.shape

(160, 31)

In [4]:
kospi.head()

,날짜,전경련BSI,주택매매가격,건설BSI(전망),"실업률(계절조정,%)",어음부도율,...,총투자율,Reuter CRB(EW)상품선물지수,Reuter CRB(EW)에너지지수,Reuter CRB(EW)산업지수,Reuter CRB(EW)귀금속지수,코스피지수
0,2003-05,96.4,70.13,83.3,3.7,0.09,...,32.3,243.7,247.7,217.5,322.4,633.4
1,2003-06,90.3,70.83,86.7,3.7,0.08,...,32.3,247.6,255.7,247.3,326.4,669.9
2,2003-07,91.4,70.57,86.7,3.8,0.06,...,31.8,249.9,268.7,232.3,340.1,713.5
3,2003-08,109.6,69.99,92.4,3.9,0.06,...,31.8,255.3,283.3,256.6,364.1,759.5
4,2003-09,110.3,69.70,64.1,3.8,0.08,...,31.8,262.6,292.7,262.0,368.3,697.5


변수 그룹 (특성 29개 + 목표변수):

| 그룹 | 변수 |
|---|---|
| 심리지표 | 전경련BSI, 건설BSI(전망), 한국(OECD) |
| 실물경제·신용 | 주택매매가격, 실업률(계절조정,%), 어음부도율, 내국인출국자수, 내국인출국자수YoY |
| 교역 | 일별수출액YoY |
| 금리·스프레드 | 통안채 364일, 공사채 AAA 3년, 은행채 AAA 3년, 국고채5년 - CD, 국고채10년 - 통안채1년, 은행채AAA3년 - 국고채3년, 회사채AA-3년 - 국고채3년 |
| 시장위험 | 코스피 200 변동성지수 |
| 국민계정 (분기 값을 월별로 반복 기재) | GDP서비스업, 설비투자, 건설투자, 재화수출, 내수, GDP디플레이터, 총저축률, 총투자율 |
| 원자재 | Reuter CRB(EW) 상품선물·에너지·산업·귀금속 지수 |
| **목표변수** | **코스피지수** (월말 종가) |

*`날짜` 열은 원본 파일에 없던 것으로, 코스피 월말 종가와 대조해 추정했습니다 (예: 2008-10 = 1,113.1, 2016-08 = 2,034.7). 영문 변수명과 설명은 `KOSPI_Data_Dictionary_KO.csv`에 있습니다.*

In [5]:
X = kospi.drop(columns=['날짜', '코스피지수'])   # 특성
y = kospi['코스피지수']                          # 목표변수
dates = pd.to_datetime(kospi['날짜'])
print(X.shape, y.shape)
y.head()

(160, 29) (160,)


0    633.4
1    669.9
2    713.5
3    759.5
4    697.5
Name: 코스피지수, dtype: float64

In [6]:
y.tail()

155    1994.2
156    1983.4
157    1970.4
158    2016.2
159    2034.7
Name: 코스피지수, dtype: float64

# 실습

## 2. 훈련/테스트 분리

In [7]:
from sklearn.model_selection import train_test_split

# 원본과 같은 설정: 무작위 80/20 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
columns = X_train.columns
X_train.head()

,전경련BSI,주택매매가격,건설BSI(전망),"실업률(계절조정,%)",어음부도율,일별수출액YoY,...,총저축률,총투자율,Reuter CRB(EW)상품선물지수,Reuter CRB(EW)에너지지수,Reuter CRB(EW)산업지수,Reuter CRB(EW)귀금속지수
60,95.3,85.62,61.6,3.3,0.02,19.08,...,32.9,34.2,452.4,873.6,378.3,646.1
115,85.7,93.31,70.8,3.1,0.02,1.62,...,34.1,28.6,541.8,879.6,481.4,1088.3
2,91.4,70.57,86.7,3.8,0.06,14.83,...,33.3,31.8,249.9,268.7,232.3,340.1
123,94.4,93.80,66.5,3.1,0.02,8.15,...,34.4,28.9,508.1,924.0,489.2,916.1
45,112.3,80.94,92.2,3.3,0.02,15.62,...,33.0,33.0,410.4,661.9,430.8,633.1


### 특성 표준화
스케일러는 **훈련 세트에서만** fit하고, 같은 변환을 테스트 세트에 적용합니다.

In [8]:
from sklearn.preprocessing import StandardScaler

std = StandardScaler()
X_train = std.fit_transform(X_train)
X_test = std.transform(X_test)   # 여기서 fit하면 안 됨: 테스트 정보가 새어 들어간다

## 3. 선형회귀

In [9]:
# 1. 인스턴스화
from sklearn.linear_model import LinearRegression
lr = LinearRegression()

In [10]:
# 2. 적합
lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [11]:
# 3. 예측
y_pred = lr.predict(X_test)

In [12]:
# 4. 평가: RMSE와 R^2
from sklearn.metrics import mean_squared_error as mse, r2_score as r2

print('RMSE: ', math.sqrt(mse(y_test, y_pred)))
print('R^2 : ', r2(y_test, y_pred) * 100)   # r2(실제값, 예측값): 순서가 중요하다!

RMSE:  107.51042716353047
R^2 :  94.6046933492504


In [13]:
# 테스트 세트 평균 수준 대비 RMSE
math.sqrt(mse(y_test, y_pred)) / np.mean(y_test)

np.float64(0.06763456634981717)

In [14]:
# 회귀계수 (표준화된 특성 기준), 크기순 정렬
coef_lr = pd.Series(lr.coef_, index=columns)
coef_lr.sort_values(key=abs, ascending=False).round(1).head(10)

은행채 AAA 3년              852.4
공사채 AAA 3년             -728.9
주택매매가격                  307.0
Reuter CRB(EW)상품선물지수    203.9
Reuter CRB(EW)귀금속지수     -90.3
내국인출국자수                  70.7
국고채5년 - CD              -59.1
회사채AA-3년 - 국고채3년        -55.1
은행채AAA3년 - 국고채3년        -48.8
통안채 364일                -41.7
dtype: float64

> **공사채 AAA 3년 ≈ −729**, **은행채 AAA 3년 ≈ +852**에 주목하세요. 두 금리는 거의 같이 움직여서(상관계수 0.997) OLS가 서로 상쇄하는 큰 계수를 줍니다. 두 계수는 하나씩 떼어 해석하지 말고 함께 봐야 합니다 — 다중공선성입니다.

## 4. 릿지 회귀

In [15]:
# 1. 인스턴스화
from sklearn.linear_model import Ridge
ridge_model = Ridge(alpha=100)

# 2. 적합
ridge_model.fit(X_train, y_train)

# 3. 예측
y_pred = ridge_model.predict(X_test)

# 4. 평가: RMSE와 R^2
print('RMSE:', math.sqrt(mse(y_test, y_pred)))
print('R^2 :', r2(y_test, y_pred) * 100)

# 회귀계수
pd.Series(ridge_model.coef_, index=columns).round(1).to_frame('릿지 계수').T

RMSE: 159.2749923751821
R^2 : 88.15840660486015


,전경련BSI,주택매매가격,건설BSI(전망),"실업률(계절조정,%)",어음부도율,일별수출액YoY,...,총저축률,총투자율,Reuter CRB(EW)상품선물지수,Reuter CRB(EW)에너지지수,Reuter CRB(EW)산업지수,Reuter CRB(EW)귀금속지수
릿지 계수,8.1,82.4,-2.6,-26.9,-51.7,-14.5,...,8.3,-3.6,53.8,37.7,36.3,46.0


## 5. 라쏘 회귀

In [16]:
# 1. 인스턴스화
from sklearn.linear_model import Lasso
lasso_model = Lasso(alpha=100)

# 2. 적합
lasso_model.fit(X_train, y_train)

# 3. 예측
y_pred = lasso_model.predict(X_test)

# 4. 평가: RMSE와 R^2
print('RMSE:', math.sqrt(mse(y_test, y_pred)))
print('R^2 :', r2(y_test, y_pred) * 100)

# 회귀계수: 라쏘는 대부분을 정확히 0으로 만든다
coef_lasso = pd.Series(lasso_model.coef_, index=columns)
coef_lasso[coef_lasso != 0].round(1)

RMSE: 173.4861535027301
R^2 : 85.95102645191307


주택매매가격                  222.4
어음부도율                   -22.7
Reuter CRB(EW)상품선물지수     85.8
dtype: float64

## 6. 다항식 회귀

In [17]:
from sklearn.preprocessing import PolynomialFeatures

# 변환기 인스턴스화 (2차: 제곱항 + 두 변수 곱의 상호작용항 + 상수열)
poly = PolynomialFeatures(degree=2)

# X 변환: fit_transform = 훈련 세트에서 fit + transform
poly_X_train = poly.fit_transform(X_train)
poly_X_test = poly.transform(X_test)   # 테스트 세트도 같은 방식으로 변환
print(poly_X_train.shape, poly_X_test.shape)   # 열은 465개인데 훈련 행은 128개

(128, 465) (32, 465)


In [18]:
lr = LinearRegression()
lr.fit(poly_X_train, y_train)
y_pred = lr.predict(poly_X_test)
print('RMSE:', math.sqrt(mse(y_test, y_pred)))
print('R^2 :', r2(y_test, y_pred))

RMSE: 129.08140380765073
R^2 : 0.9222245883465182


## 7. 해결책? 다항식 특성에 라쏘와 릿지

In [19]:
lasso_model = Lasso(alpha=100)
lasso_model.fit(poly_X_train, y_train)
y_pred = lasso_model.predict(poly_X_test)
print('RMSE:', math.sqrt(mse(y_test, y_pred)))
print('R^2 :', r2(y_test, y_pred) * 100)
print('0이 아닌 계수:', np.sum(lasso_model.coef_ != 0), '/', lasso_model.coef_.size)

RMSE: 169.74520028255964
R^2 : 86.55038153025177
0이 아닌 계수: 7 / 465


In [20]:
ridge_model = Ridge(alpha=1)
ridge_model.fit(poly_X_train, y_train)
y_pred = ridge_model.predict(poly_X_test)
print('RMSE:', math.sqrt(mse(y_test, y_pred)))
print('R^2 :', r2(y_test, y_pred) * 100)

RMSE: 124.69766581398255
R^2 : 92.7417552790614


## 8. 요약 (무작위 분리)

In [21]:
def evaluate(models, Xtr, Xte, ytr, yte, Ptr=None, Pte=None):
    rows = []
    for name, model, use_poly in models:
        a, b = (Ptr, Pte) if use_poly else (Xtr, Xte)
        model.fit(a, ytr)
        yp = model.predict(b)
        rows.append({'모델': name,
                     '테스트 RMSE': math.sqrt(mse(yte, yp)),
                     '테스트 R2': r2(yte, yp),
                     '훈련 R2': r2(ytr, model.predict(a)),
                     '0이 아닌 계수': int(np.sum(np.abs(model.coef_) > 1e-9))})
    return pd.DataFrame(rows).set_index('모델').round(3)

MODELS = lambda: [('선형회귀', LinearRegression(), False),
                  ('릿지 a=100', Ridge(alpha=100), False),
                  ('라쏘 a=100', Lasso(alpha=100), False),
                  ('2차 다항식 + OLS', LinearRegression(), True),
                  ('2차 다항식 + 라쏘 a=100', Lasso(alpha=100), True),
                  ('2차 다항식 + 릿지 a=1', Ridge(alpha=1), True)]

res_random = evaluate(MODELS(), X_train, X_test, y_train, y_test, poly_X_train, poly_X_test)
res_random

,테스트 RMSE,테스트 R2,훈련 R2,0이 아닌 계수
모델,,,,
선형회귀,107.510,0.946,0.968,29
릿지 a=100,159.275,0.882,0.909,29
라쏘 a=100,173.486,0.860,0.835,3
2차 다항식 + OLS,129.081,0.922,1.000,464
2차 다항식 + 라쏘 a=100,169.745,0.866,0.850,7
2차 다항식 + 릿지 a=1,124.698,0.927,1.000,464


무작위 분할에서 모든 모델의 테스트 R²가 0.85를 넘고, 테스트 RMSE는 단순 OLS가 가장 낮습니다.